# Randomness Checker

Here let's use kolmogorov-smirnov, chi-square, and poker tests to check the randomness of the generated numbers. Randomness tests are used to check the properties like uniformity, independence, and distribution of the generated numbers.

## Utility

We will use python's `random` module to generate random numbers.

In [1]:
import random, math

random_numbers = [random.random() for _ in range(10)]
random_10000 = [random.random() * 10000 for _ in range(10000)]
random_100 = [random.random() * 100 for _ in range(100)]
random_90 = [random.random() * 90 for _ in range(100)]

## Kolmogorov - Smirnov test

In [2]:
def kolmogorov(random_numbers):
    n = len(random_numbers)
    sorted_numbers = sorted(random_numbers)
    
    d_plus = -math.inf
    for i in range(n):
        d_plus = max(d_plus, (i + 1) / n - sorted_numbers[i])

    d_minus = -math.inf
    for i in range(n):
        d_minus = max(d_minus, sorted_numbers[i] - i / n)

    d_statistic = max(d_plus, d_minus)
    return d_statistic

print(kolmogorov(random_numbers))

0.2023923756263653


## Chi - Square test

In [3]:
def chi_square_test(random_numbers, k):
    n = len(random_numbers)
    expected_count = n / k
    observed_counts = [0] * k

    class_ranges = [i * k for i in range(1, k + 1)]

    for number in random_numbers:
        for i in class_ranges:
            if number <= i:
                observed_counts[class_ranges.index(i)] += 1
                break

    print("Observed Counts:", observed_counts)
    chi_square_statistic = 0
    for i in observed_counts:
        chi_square_statistic += (i - expected_count) ** 2 
    return chi_square_statistic / expected_count

print(chi_square_test(random_100, 10))

Observed Counts: [12, 7, 8, 8, 12, 7, 8, 13, 15, 10]
7.2


## Auto - Correlation test

In [4]:
def auto_correlation(random_numbers):
    # Group 3
    n = len(random_numbers)
    expected_count = n / 9
    observed_counts = [0] * 9

    for i in range(n - 1):
        r1 = random_numbers[i]
        r2 = random_numbers[i + 1]

        if r1 <= 33 and r2 <= 33:
            observed_counts[0] += 1
        elif r1 <= 67 and r2 <= 33:
            observed_counts[1] += 1
        elif r1 <= 100 and r2 <= 33:
            observed_counts[2] += 1
        elif r1 <= 33 and r2 <= 67:
            observed_counts[3] += 1
        elif r1 <= 67 and r2 <= 67:
            observed_counts[4] += 1
        elif r1 <= 100 and r2 <= 67:
            observed_counts[5] += 1
        elif r1 <= 33 and r2 <= 100:
            observed_counts[6] += 1
        elif r1 <= 67 and r2 <= 100:
            observed_counts[7] += 1
        elif r1 <= 100 and r2 <= 100:
            observed_counts[8] += 1

    statistic = 0
    for i in observed_counts:
        statistic += (i - expected_count) ** 2 / expected_count

    return statistic

print(auto_correlation(random_90))

16.39


## Poker Test

In [9]:
def calculate_poker_distribution(random_numbers):
    distribution = [0] * 7
    for number in random_numbers:
        digits = sorted(str(int(number)).zfill(5))
        unique_digits = set(digits)
        if len(unique_digits) == 1:
            distribution[0] += 1  # Five of a kind
        elif len(unique_digits) == 2:
            if any(digits.count(digit) == 4 for digit in unique_digits):
                distribution[1] += 1  # Four of a kind
            else:
                distribution[2] += 1  # Full house
        elif len(unique_digits) == 3:
            if any(digits.count(digit) == 3 for digit in unique_digits):
                distribution[3] += 1  # Three of a kind
            else:
                distribution[4] += 1  # Two pairs
        elif len(unique_digits) == 4:
            distribution[5] += 1  # One pair
        else:
            distribution[6] += 1  # All different

    return distribution

def poker_test(random_numbers):
    expected_distribution = calculate_poker_distribution([i for i in range(10000)])
    observed_distribution = calculate_poker_distribution(random_numbers)

    statistic = 0
    for i in range(len(expected_distribution)):
        statistic += (observed_distribution[i] - expected_distribution[i]) ** 2 / expected_distribution[i]
    return statistic

print(poker_test(random_10000))

1.5173280423280424
